# small128 iter-1 TAKE-3 — peer-review consensus recipe

**Same corpus, same base — new loss.** Take-2 (dw3/T0.7/lr3e-4/bs32k, pure soft-CE) regressed:
converged by ep7 with corrections starved (dw3 gave disagreement states 0.36× mean weight — on a
uniformly-sharp corpus, contested visits = low top-share = down-weighted) and 8.4% collateral
argmax drift on agreement states (flatter-than-base soft targets, no hard anchor).
Diagnosis brief: `docs/small128_iter1_for_review.md`. Gemini + ChatGPT converged on this fix:

- **`--decisiveness-power 0`** — dw3 is structurally hostile to corrections against a strong prior.
- **`--blend-alpha 0.5`** — restore the hard-CE blend that BUILT ep87: hard-CE toward the target
  argmax anchors agreements (kills drift) and decisively pulls disagreements (the corrections).
- **`--target-temperature 1.0`** — no sharpening of already-sharp targets.
- **`--batch-size 4096 --lr 1e-4`** — the base's native optimization regime (~5,800 steps/epoch).

**GATE (rejection rule): eval ep1 vs ep87 — floor first. ep1 < base ⇒ recipe rejected, stop.**
Bar (500 seeds 775000-775499, fp16): mean 12,987 / P50 8,889 / P5 1,082 / P10 1,799 / <1000 4.8%.

Drive needs (all already uploaded): `colorlines_pillar3d_v2.tar.gz`, `small128_iter1.pt.gz`
(231,393,038 B), `pillar3k_small128_hardce_epoch_87.pt`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, time
DRIVE='/content/drive/MyDrive/alphatrain'
!cp {DRIVE}/colorlines_pillar3d_v2.tar.gz /content/
!cd /content && tar xzf colorlines_pillar3d_v2.tar.gz
os.makedirs('/content/alphatrain/data', exist_ok=True)
t0=time.time()
!cp {DRIVE}/small128_iter1.pt.gz /content/small128_iter1.pt.gz
gz=os.path.getsize('/content/small128_iter1.pt.gz'); print(f'.gz: {gz:,} bytes')
assert gz == 231_393_038, f'.gz truncated! got {gz}; re-upload small128_iter1.pt.gz'
!gunzip -t /content/small128_iter1.pt.gz && echo '.gz integrity OK'
!gzip -dc /content/small128_iter1.pt.gz > /content/alphatrain/data/small128_iter1.pt
pt=os.path.getsize('/content/alphatrain/data/small128_iter1.pt')
assert pt == 1_139_353_549, f'.pt size wrong! got {pt}'
print(f'corpus: {pt/1e9:.2f} GB, 2,974,799 states ({time.time()-t0:.0f}s)')
!rm /content/small128_iter1.pt.gz
!cp {DRIVE}/pillar3k_small128_hardce_epoch_87.pt /content/alphatrain/data/
!pip install -q numpy numba scipy

In [ ]:
import torch
print(f'PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g=torch.cuda.get_device_properties(0); print(f'GPU {torch.cuda.get_device_name(0)} | {g.total_memory/1e9:.0f} GB')

In [ ]:
# ===== CONFIG (peer-review consensus; one variable philosophy: flags only, no code change) =====
CHANNELS = 128
EPOCHS   = 8         # gate at ep1; ChatGPT suggested 5, headroom to 8 with per-epoch saves
BATCH    = 4096      # the base's native regime (~5,800 steps/epoch)
LR       = 1e-4
T        = 1.0       # NO sharpening
DW       = 0         # dw3 DROPPED (inverted selectivity on this corpus)
BLEND    = 0.5       # 0.5*soft-CE + 0.5*hard-CE(target argmax) — the loss that built ep87
RUN      = "small128_iter1t3"
print(f'RUN={RUN} ch={CHANNELS} epochs={EPOCHS} batch={BATCH} lr={LR} T={T} dw={DW} blend={BLEND}')

In [ ]:
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train_path_b \
    --tensor-file alphatrain/data/small128_iter1.pt \
    --resume alphatrain/data/pillar3k_small128_hardce_epoch_87.pt --warm-start \
    --channels {CHANNELS} --amp --compile \
    --epochs {EPOCHS} --batch-size {BATCH} --lr {LR} --warmup-epochs 1 \
    --target-temperature {T} --decisiveness-power {DW} --blend-alpha {BLEND} \
    --copy-to /content/drive/MyDrive/alphatrain/{RUN}_best.pt \
    --save-dir /content/checkpoints/{RUN} 2>&1 | tee /content/{RUN}_train.log

In [ ]:
import shutil, os, glob
DRIVE='/content/drive/MyDrive/alphatrain'
for f in sorted(glob.glob(f'/content/checkpoints/{RUN}/epoch_*.pt')):
    dst=f'{DRIVE}/{RUN}_{os.path.basename(f)}'; shutil.copy(f,dst); print('Saved', dst)
for f in ['best.pt','latest.pt']:
    s=f'/content/checkpoints/{RUN}/{f}'
    if os.path.exists(s): shutil.copy(s,f'{DRIVE}/{RUN}_{f}'); print('Saved', f'{DRIVE}/{RUN}_{f}')

## Gate protocol (M5, C++ eval)

1. **ep1 FIRST** (download `small128_iter1t3_epoch_1.pt`):
```bash
python -m alphatrain.inference_cpp.export_ts --model alphatrain/data/small128_iter1t3_epoch_1.pt
cd alphatrain/inference_cpp
./build/eval --model data/policy_ts.pt --device mps --seed-start 775000 --seed-end 775500 --batch 500
```
   **ep1 floor < ep87 floor ⇒ recipe REJECTED, stop the run** (rejection gate, not proof of health).
2. If ep1 ≥ base: eval ep3/ep5/ep8, pick by floor (P5/P10/<1000), 5k range for close calls.
3. Also re-run the adoption probe on the best epoch — healthy signature = adoption well above 25%
   with agreement-state retention ~99% (was 91.6% in take-2).
4. Escalation if the gate fails: split-gradient audit results + γ-disagreement weighting
   (w = 1 + γ·disagree, needs a small trainer change) or Gemini's agreement-downsampling — take-4.